# CARE-Fusion — Multi-dataset transformer run (A100)

Chạy transformer trên **3 dataset**: ViGoEmotions (VN, PhoBERT), UIT-VSMEC (VN, PhoBERT), GoEmotions (EN, XLM-R).
Mỗi dataset: preprocess → q/PMI → OOF → weak labels → experiments (baselines + CARE) → emoji-amplified (B1/B6) → slice summary.

**Runtime → Change runtime type → A100 GPU.** Ước ~nhiều giờ; kết quả lưu ra Drive. Chạy lần lượt từng ô.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!git clone https://github.com/dtlong1979/care-fusion-sentiment-detection-vi.git
%cd care-fusion-sentiment-detection-vi

In [ ]:
# Java cho VnCoreNLP (segment tiếng Việt) + deps
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
import os; os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
!pip install -q -r requirements.txt
!pip install -q -e .

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os; os.makedirs('/content/drive/MyDrive/care-fusion', exist_ok=True)

In [ ]:
# Tải 2 dataset bổ sung + chuyển về schema thống nhất (ViGoEmotions đã có trong repo)
B='https://raw.githubusercontent.com/dinhthienan33/RecognitionEmotion-for-Vietnamese-Social-Media/main/UIT-VSMEC'
!mkdir -p data/vsmec_raw && for f in train valid test; do wget -q -P data/vsmec_raw $B/$f.csv; done
G='https://storage.googleapis.com/gresearch/goemotions/data/full_dataset'
!mkdir -p data/goemotions_raw && for i in 1 2 3; do wget -q -P data/goemotions_raw $G/goemotions_$i.csv; done
!python scripts/prepare_extra_datasets.py

In [ ]:
# 1) ViGoEmotions (VN, PhoBERT) — đầy đủ baseline + ablation
OUT='/content/drive/MyDrive/care-fusion/vigoemotions'
V='B0_majority,B1_text,B2_concat,B3_gated,B4_crossattn,CARE_full,CARE_-routing,CARE_-delta,CARE_-cf,CARE_-intensity,CARE_-gcn,CARE_-confidence'
!python scripts/run_all_datasets.py --config configs/default.yaml --out $OUT --variants $V

In [ ]:
# 2) UIT-VSMEC (VN, PhoBERT) — baseline + CARE
OUT='/content/drive/MyDrive/care-fusion/vsmec'
!python scripts/run_all_datasets.py --config configs/vsmec.yaml --out $OUT

In [ ]:
# 3) GoEmotions (EN, XLM-R) — baseline + CARE
OUT='/content/drive/MyDrive/care-fusion/goemotions'
!python scripts/run_all_datasets.py --config configs/goemotions.yaml --out $OUT

In [ ]:
# Xong: tự ngắt để khỏi tốn CU
print('>>> ALL DONE <<<')
from google.colab import runtime; runtime.unassign()